# IRC-Bench Fine-Tuning: DPR Bi-Encoder + Cross-Encoder

This notebook runs the full fine-tuning pipeline for the IRC (Implicit Reference Completion) benchmark.

**Models trained:**
1. DPR bi-encoder with BGE-base
2. DPR bi-encoder with MiniLM (comparison)
3. Cross-encoder with MS-MARCO MiniLM
4. (Optional) LoRA fine-tuned LLM

**Requirements:** GPU runtime (T4 or better). Go to Runtime > Change runtime type > GPU.

In [ ]:
# Clone repo and install deps
!git clone https://github.com/ApartsinProjects/ImplicitEntities.git
%cd ImplicitEntities
!pip install -q sentence-transformers pandas numpy

In [ ]:
import torch
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# DPR Training with BGE-base (primary model)
!cd experiments && python train_dpr.py \
    --base-model BAAI/bge-base-en-v1.5 \
    --mode description \
    --epochs 10 \
    --batch-size 64 \
    --lr 2e-5

In [ ]:
# DPR Training with MiniLM (comparison baseline)
!cd experiments && python train_dpr.py \
    --base-model all-MiniLM-L6-v2 \
    --mode description \
    --epochs 10 \
    --batch-size 64

In [ ]:
# Cross-Encoder Training
!cd experiments && python train_crossencoder.py \
    --mode train \
    --base-model cross-encoder/ms-marco-MiniLM-L-6-v2 \
    --epochs 5 \
    --batch-size 32 \
    --entity-repr description

In [ ]:
# Evaluate DPR models
!cd experiments && python train_dpr.py --eval-only --model-path trained_models/dpr_description_BAAI_bge-base-en-v1.5
!cd experiments && python train_dpr.py --eval-only --model-path trained_models/dpr_description_all-MiniLM-L6-v2

# Evaluate cross-encoder
!cd experiments && python train_crossencoder.py --mode eval --model-path trained_models/crossencoder_ms-marco-MiniLM-L-6-v2_*

In [ ]:
# Save results to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp -r experiments/results/ /content/drive/MyDrive/IRC_results/
!cp -r experiments/trained_models/ /content/drive/MyDrive/IRC_models/
print("Results saved to Google Drive")

## LoRA Fine-Tuning (Optional)

Requires more VRAM than a T4 provides. Use an A100 or L4 runtime if available.
This cell fine-tunes a small LLM (Llama 3.2 1B) with QLoRA for the entity resolution task.

In [ ]:
# LoRA fine-tuning (optional, needs A100/L4 runtime)
!pip install -q peft bitsandbytes accelerate trl
!cd experiments && python train_lora_llm.py \
    --base-model meta-llama/Llama-3.2-1B-Instruct \
    --epochs 3 \
    --batch-size 4 \
    --gradient-accumulation 4 \
    --quantize 4bit